# Notebook 2 - SAR Preprocessing
**Inputs:** Sentinel-1 GRD via Element84 STAC  
**Outputs:** data/processed/sar/YYYY-MM_VV.tif and YYYY-MM_VH.tif (COG)

## Steps
1. Load Sentinel-1 VV/VH per month via odc-stac
2. Monthly median composite
3. Lee speckle filter
4. Convert to dB
5. Save monthly COGs
6. Build 3-month dry baseline

## Key Limitations
- Sentinel-1 revisit ~12 days: floods between passes may be missed
- Lee 7x7 filter smooths fine-scale flood boundaries
- Virunga/Ruwenzori terrain causes SAR layover/shadow artifacts


In [ ]:
# Clone repo (Colab only)
import os
if not os.path.exists("Floodmaps"):
    !git clone https://github.com/trevmon28/Floodmaps.git
os.chdir("Floodmaps")
print("Working directory:", os.getcwd())


In [ ]:
import subprocess, sys
packages = [
    "rasterio", "rioxarray", "xarray", "dask[distributed]",
    "pystac-client", "geopandas", "shapely",
    "scipy", "scikit-image", "numpy", "pyyaml", "pyproj",
    "tqdm", "odc-stac", "odc-geo"
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + packages)
print("Packages ready.")


In [ ]:
import yaml, os, warnings
import numpy as np
import rioxarray
import rasterio
from rasterio.shutil import copy as rio_copy
import pystac_client
import pandas as pd
from scipy.ndimage import uniform_filter
from tqdm import tqdm
import odc.stac
warnings.filterwarnings("ignore")
os.environ["AWS_NO_SIGN_REQUEST"] = "YES"

with open("config/config.yaml") as f:
    cfg = yaml.safe_load(f)

b = cfg["aoi"]["bbox"]
BBOX = [b["west"], b["south"], b["east"], b["north"]]
OUTPUT_CRS = cfg["aoi"]["output_crs"]
PROCESSED_DIR = "data/processed/sar"
BASELINE_MONTHS = cfg["processing"]["baseline_months"]
os.makedirs(PROCESSED_DIR, exist_ok=True)
print("Config loaded. BBOX:", BBOX)


In [ ]:
def monthly_date_ranges(cfg):
    start = pd.Timestamp(cfg["temporal"]["start"])
    end = pd.Timestamp(cfg["temporal"]["end"])
    months = pd.date_range(start, end, freq="MS")
    return [(str(m.date()), str((m + pd.offsets.MonthEnd(1)).date())) for m in months]

def to_db(arr):
    return 10 * np.log10(np.where(arr > 0, arr, np.nan))

def lee_filter(arr, size=7):
    img = arr.astype("float64")
    img_mean = uniform_filter(img, size)
    img_sq_mean = uniform_filter(img ** 2, size)
    img_var = img_sq_mean - img_mean ** 2
    overall_var = np.nanvar(img)
    weights = img_var / (img_var + overall_var + 1e-10)
    return img_mean + weights * (img - img_mean)

def write_cog(array, profile, output_path):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    tmp = output_path + ".tmp.tif"
    profile.update(driver="GTiff", compress="deflate", tiled=True,
                   blockxsize=512, blockysize=512, dtype="float32", count=1)
    with rasterio.open(tmp, "w", **profile) as dst:
        dst.write(array.astype("float32"), 1)
    with rasterio.open(tmp, "r+") as dst:
        dst.build_overviews([2, 4, 8, 16], rasterio.enums.Resampling.average)
        dst.update_tags(ns="rio_overview", resampling="average")
    rio_copy(tmp, output_path, driver="GTiff", copy_src_overviews=True,
             compress="deflate", tiled=True, blockxsize=512, blockysize=512)
    os.remove(tmp)
    print("  Saved:", output_path)

print("Helpers defined.")


In [ ]:
catalog = pystac_client.Client.open("https://earth-search.aws.element84.com/v1")
date_ranges = monthly_date_ranges(cfg)
print(len(date_ranges), "monthly windows")
print("First:", date_ranges[0], "| Last:", date_ranges[-1])


In [ ]:
process_all = False
month_idx = 0
# Set resolution=20 for production (requires overnight run)
RESOLUTION = 100

months_to_run = date_ranges if process_all else [date_ranges[month_idx]]
monthly_vv = {}

for start, end in tqdm(months_to_run, desc="Processing months"):
    month_str = start[:7]
    out_vv = f"{PROCESSED_DIR}/{month_str}_VV.tif"
    out_vh = f"{PROCESSED_DIR}/{month_str}_VH.tif"

    if os.path.exists(out_vv) and os.path.exists(out_vh):
        print(f"  {month_str} already processed - skipping")
        continue

    print(f"Searching Sentinel-1 for {month_str}...")
    results = catalog.search(
        collections=["sentinel-1-grd"],
        bbox=BBOX,
        datetime=f"{start}/{end}",
    )
    items = list(results.items())
    print(f"  Found {len(items)} scenes")

    if not items:
        print(f"  No scenes for {month_str} - skipping")
        continue

    try:
        ds = odc.stac.load(
            items,
            bands=["vv", "vh"],
            bbox=BBOX,
            resolution=RESOLUTION,
            crs=OUTPUT_CRS,
            dtype="float64",
            groupby="solar_day",
            chunks={"x": 2048, "y": 2048},
        )
        print(f"  Loaded: {dict(ds.dims)}")
    except Exception as e:
        print(f"  Load failed for {month_str}: {e}")
        continue

    composite = ds.median(dim="time")

    for band_name in ["vv", "vh"]:
        arr = composite[band_name].compute().values
        arr = lee_filter(arr)
        arr = to_db(arr)
        profile = {
            "crs": OUTPUT_CRS,
            "transform": composite.rio.transform(),
            "width": arr.shape[1],
            "height": arr.shape[0],
            "nodata": float("nan"),
        }
        out_path = f"{PROCESSED_DIR}/{month_str}_{band_name.upper()}.tif"
        write_cog(arr, profile, out_path)
        if band_name == "vv":
            monthly_vv[month_str] = arr

print("Done.")


In [ ]:
BASELINE_MONTHS = 1  # override for single-month test; remove when process_all=True
baseline_path = f"{PROCESSED_DIR}/baseline_VV.tif"

if not os.path.exists(baseline_path):
    files = sorted([f for f in os.listdir(PROCESSED_DIR) if f.endswith("_VV.tif")])[:BASELINE_MONTHS]
    if len(files) < BASELINE_MONTHS:
        print(f"Only {len(files)} months ready - need {BASELINE_MONTHS} for baseline")
    else:
        arrays, profile = [], None
        for fname in files:
            with rasterio.open(f"{PROCESSED_DIR}/{fname}") as src:
                arrays.append(src.read(1).astype("float32"))
                if profile is None:
                    profile = src.profile.copy()
        baseline = np.nanmedian(np.stack(arrays), axis=0)
        write_cog(baseline, profile, baseline_path)
        print("Baseline saved from:", files)
else:
    print("Baseline already exists:", baseline_path)


In [ ]:
import matplotlib.pyplot as plt

vv_files = sorted([f for f in os.listdir(PROCESSED_DIR) if f.endswith("_VV.tif")])
if vv_files:
    with rasterio.open(f"{PROCESSED_DIR}/{vv_files[0]}") as src:
        vv_db = src.read(1)
    plt.figure(figsize=(12, 8))
    plt.imshow(vv_db, cmap="gray", vmin=-25, vmax=0)
    plt.colorbar(label="Backscatter (dB)")
    plt.title(f"Sentinel-1 VV - {vv_files[0]}")
    plt.axis("off")
    plt.show()
    print(f"Range: {np.nanmin(vv_db):.1f} to {np.nanmax(vv_db):.1f} dB")
else:
    print("No processed files yet.")
